In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import scipy.sparse as sp

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_fscore_support,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

In [ ]:
DATA_PATH = Path("/home/ubuntu/MLLM-interpretability/data/processed/training_data.csv")
TARGET = "correct_target"

In [ ]:
MODEL1_FEATURES = ["source_lang", "target_lang"]
TEXT_COLS = ["q_src", "q_tgt", "a_src"]
RANDOM_STATE = 42
TEST_SIZE = 0.2

In [ ]:
ALL_FEATURES = ['language_version_count', 'qa_topic','question_type', 'cooc_num_pairs', 'cooc_total_pairs', 'cooc_coverage_ratio', 'cooc_unseen_keywords_count', 'cooc_unseen_keywords_ratio', 'cooc_avg_pmi', 'cooc_max_pmi', 'cooc_min_pmi', 'cooc_std_pmi', 'target_avg_dep_depth', 'target_max_tree_depth', 'target_num_clauses', 'source_avg_dep_depth', 'source_max_tree_depth', 'source_num_clauses', 'source_family', 'source_genus', 'target_family', 'target_genus', 'source_script', 'source_syllables', 'source_wiki_size', 'target_script', 'target_syllables', 'target_wiki_size']
qa_topic_feat = 'qa_topic'
language_version_count_feat = 'language_version_count'
question_type_feat = 'question_type'
cooc_features = ['cooc_num_pairs', 'cooc_total_pairs', 'cooc_coverage_ratio', 'cooc_unseen_keywords_count', 'cooc_unseen_keywords_ratio', 'cooc_avg_pmi', 'cooc_max_pmi', 'cooc_min_pmi', 'cooc_std_pmi']
syntactic_features = ['target_avg_dep_depth', 'target_max_tree_depth', 'target_num_clauses', 'source_avg_dep_depth', 'source_max_tree_depth', 'source_num_clauses']
linguistic_features = ['source_family', 'source_genus', 'target_family', 'target_genus', 'source_script', 'source_syllables', 'target_script', 'target_syllables']
wiki_size_features = ['source_wiki_size', 'target_wiki_size']

In [ ]:
MODEL2_FEATURES = ALL_FEATURES

In [ ]:
def split_features(df, feature_list):
    X = df[feature_list].copy()
    y = df[TARGET].copy()
    # Map common string/bool to 0/1 if needed
    if y.dtype == "O":
        y = y.map({True: 1, False: 0, "true": 1, "false": 0, "yes": 1, "no": 0, "1": 1, "0": 0})
    y = y.astype(float).astype(int)
    return X, y

def _to_csr(X):
    # Utility to force dense -> sparse CSR (helps keep global feature matrix sparse)
    if sp.issparse(X):
        return X
    return sp.csr_matrix(X)

def build_preprocess_pipe(X: pd.DataFrame, text_features=None) -> ColumnTransformer:
    """Create a ColumnTransformer for numeric, categorical, and optional text columns."""
    text_features = text_features or []
    present_text = [c for c in text_features if c in X.columns]

    numeric_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c]) and c not in present_text]
    categorical_cols = [c for c in X.columns if c not in numeric_cols and c not in present_text]

    # Numeric: impute median, scale (with_mean=False to play nice with potential sparsity), then convert to CSR
    num_pipe = Pipeline(
        [
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler(with_mean=False)),
            ("to_csr", FunctionTransformer(_to_csr, accept_sparse=True)),
        ]
    )

    # Categorical: impute mode, OHE (newer sklearn uses 'sparse_output' instead of 'sparse')
    cat_pipe = Pipeline(
        [
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
        ]
    )

    transformers = []
    if numeric_cols:
        transformers.append(("num", num_pipe, numeric_cols))
    if categorical_cols:
        transformers.append(("cat", cat_pipe, categorical_cols))

    # Optional per-text-column TF-IDF encoders (word-level 1-2 grams)
    for col in present_text:
        text_pipe = Pipeline(
            [
                ("fillna", FunctionTransformer(lambda s: s.fillna(""), feature_names_out="one-to-one")),
                ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=30000)),
            ]
        )
        transformers.append((f"tfidf_{col}", text_pipe, col))

    pre = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False,
        # Let sklearn decide sparse vs dense based on density; we bias toward sparse via CSR conversions above
    )
    return pre

def _feature_names_from_preprocessor(preprocessor) -> list:
    names = []
    for name, trans, cols in preprocessor.transformers_:
        if name == "remainder" and trans == "drop":
            continue
        if hasattr(trans, "get_feature_names_out"):
            # cat/num pipelines won't expose names directly, but OneHotEncoder and TfidfVectorizer do
            try:
                fn = trans.get_feature_names_out()
                names.extend(fn.tolist())
                continue
            except Exception:
                pass
        # Pipelines: try to pull last step feature names (e.g., OHE or TF-IDF)
        if hasattr(trans, "steps"):
            last = trans.steps[-1][1]
            if hasattr(last, "get_feature_names_out"):
                try:
                    base = cols if isinstance(cols, list) else [cols]
                    fn = last.get_feature_names_out(base)
                    names.extend(fn.tolist())
                    continue
                except Exception:
                    try:
                        fn = last.get_feature_names_out()
                        names.extend(fn.tolist())
                        continue
                    except Exception:
                        pass
        # Fallback: just use column names
        if isinstance(cols, list):
            names.extend(cols)
        else:
            names.append(cols)
    return names

def inspect_coefficients(pipeline: Pipeline) -> pd.DataFrame:
    """Return a DataFrame with feature, coefficient, odds ratio, and absolute coefficient."""
    pre = pipeline.named_steps["pre"]
    clf = pipeline.named_steps["clf"]
    # LogisticRegression in sklearn exposes coef_. Shape (1, n_features) for binary
    coef = np.asarray(clf.coef_).ravel()
    try:
        feat_names = pre.get_feature_names_out().tolist()
    except Exception:
        feat_names = _feature_names_from_preprocessor(pre)
    df_coef = pd.DataFrame(
        {
            "feature": feat_names,
            "coef": coef,
        }
    )
    df_coef["odds_ratio"] = np.exp(df_coef["coef"])
    df_coef["abs_coef"] = np.abs(df_coef["coef"])
    df_coef = df_coef.sort_values("abs_coef", ascending=False).reset_index(drop=True)
    return df_coef

def train_and_evaluate(df, feature_list, model_name, add_text_features=False, random_state=42):
    # Optionally extend features with text columns that exist
    if add_text_features:
        available_text = [c for c in TEXT_COLS if c in df.columns]
        features = feature_list + [c for c in available_text if c not in feature_list]
        used_text = available_text
    else:
        features = feature_list
        used_text = []

    X, y = split_features(df, features)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=random_state
    )

    pre = build_preprocess_pipe(X_train, text_features=used_text)

    clf = LogisticRegression(
        penalty="l2",
        solver="saga",
        max_iter=5000,
        n_jobs=-1,
        class_weight="balanced",
        random_state=random_state,
    )

    pipe = Pipeline(steps=[("pre", pre), ("clf", clf)])
    pipe.fit(X_train, y_train)

    proba = pipe.predict_proba(X_test)[:, 1]
    y_pred = (proba >= 0.5).astype(int)

    roc_auc = roc_auc_score(y_test, proba) if len(np.unique(y_test)) > 1 else np.nan
    pr_auc = average_precision_score(y_test, proba) if len(np.unique(y_test)) > 1 else np.nan
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="binary", zero_division=0
    )
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, zero_division=0)

    summary = {
        "model": model_name,
        "n_train": int(len(y_train)),
        "n_test": int(len(y_test)),
        "positive_rate_train": float(np.mean(y_train)),
        "positive_rate_test": float(np.mean(y_test)),
        "ROC_AUC": float(roc_auc) if not np.isnan(roc_auc) else None,
        "PR_AUC": float(pr_auc) if not np.isnan(pr_auc) else None,
        "Precision": float(precision),
        "Recall": float(recall),
        "F1": float(f1),
        "TN": int(cm[0, 0]),
        "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]),
        "TP": int(cm[1, 1]),
    }

    coef_df = inspect_coefficients(pipe)
    return {
        "pipeline": pipe,
        "summary": summary,
        "report_text": report,
        "coef_df": coef_df,
    }

In [ ]:
df = pd.read_csv(DATA_PATH)
results = []

In [ ]:
res1 = train_and_evaluate(df, MODEL1_FEATURES, "Model 1: (source_lang, target_lang)", add_text_features=False)
print("\n=== Model 1: (source_lang, target_lang) ===")
print("Summary:", res1["summary"])
print("\nClassification Report:\n", res1["report_text"])
# coef1_path = Path("./model1_coefficients.csv")
# res1["coef_df"].to_csv(coef1_path, index=False)
# print(f"Saved Model 1 coefficients to: {coef1_path}")

In [ ]:
res2 = train_and_evaluate(df, MODEL2_FEATURES, "Model 2: (rich feature set)", add_text_features=False)
print("\n=== Model 2: (rich feature set) ===")
print("Summary:", res2["summary"])
print("\nClassification Report:\n", res2["report_text"])
# coef2_path = Path("./model2_coefficients.csv")
# res2["coef_df"].to_csv(coef2_path, index=False)
# print(f"Saved Model 2 coefficients to: {coef2_path}")

In [ ]:
pipe1.named_steps["clf"].coef_

In [ ]:
pipe1.named_steps["pre"].get_feature_names_out()

In [ ]:
res2t = train_and_evaluate(
            df,
            MODEL2_FEATURES,
            "Model 2 (+ text: q_src/q_tgt/a_src where present)",
            add_text_features=True,
        )
print("\n=== Model 2 (+ text) ===")
print("Summary:", res2t["summary"])
print("\nClassification Report:\n", res2t["report_text"])
# coef2t_path = Path("./model2_plus_text_coefficients.csv")
# res2t["coef_df"].to_csv(coef2t_path, index=False)
# print(f"Saved Model 2 + text coefficients to: {coef2t_path}")

In [ ]:
# Model 1: 0.60
# Model 2: 0.61
# Model 2 (delete tree-related features): 0.6401137980085347
# Model 2 (delete cooc-related features based on previous): 0.6090564248458985
# Model 3: 0.62
# Model 3 (delete tree-related features): 0.6595542911332385
# Model 3 (delete cooc-related features based on previous): 0.6548127074442863